# Minimum-overlap MRI training inputs

Dataset V2 retains the same pixel recipe and labels, with up to 60 triplets and globally minimum repeated slice uses. Original dataset V1 is preserved. Images are ready float16 tensors. Gold74 is held out. This notebook validates and loads data only; it does not train or preprocess pixels.


In [ ]:
"""Load direct-384 preprocessed images without any pixel transformation."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


class MinimumOverlap384Dataset(Dataset):
    def __init__(self,roots,split='train',k=None,labels=True):
        roots=[Path(p) for p in roots]
        recipes=[json.loads((p/'recipe.json').read_text()) for p in roots]
        if not recipes or any(r['format']!='rsna_min_overlap_384_v1' for r in recipes) or len({r['sha256'] for r in recipes})!=1:
            raise ValueError('Attach only the new minimum-overlap direct-384 recipe')
        if k is not None and (not isinstance(k,int) or k<1):raise ValueError('K must be positive or None')
        splits=pd.concat([pd.read_csv(p/'splits.csv',dtype={'StudyInstanceUID':str}) for p in roots],ignore_index=True)
        if splits.StudyInstanceUID.duplicated().any():raise ValueError('Duplicate shards/study IDs')
        wanted=set(splits.loc[splits['split']==split,'StudyInstanceUID'])
        self.files=sorted(p for root in roots for p in root.rglob('*.npz') if p.stem in wanted)
        if {p.stem for p in self.files}!=wanted or len(self.files)!=len(wanted):raise ValueError('Missing or duplicate study files')
        self.k=k;self.labels=None
        if labels:
            if split!='train':raise ValueError('Gold labels remain separate; use labels=False for evaluation images')
            table=pd.concat([pd.read_csv(p/'labels_hybrid_train.csv',dtype={'StudyInstanceUID':str}) for p in roots],ignore_index=True)
            findings=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
            if table.StudyInstanceUID.duplicated().any() or wanted!=set(table.StudyInstanceUID) or not np.isfinite(table[findings].to_numpy()).all():
                raise ValueError('Invalid HYBRID training labels')
            self.labels=table.set_index('StudyInstanceUID')[findings]

    def __len__(self):return len(self.files)

    def __getitem__(self,index):
        file=self.files[index]
        with np.load(file,allow_pickle=False) as saved:
            images=torch.from_numpy(saved['images'])
            series=saved['triplet_series_uids']
            indices=saved['triplet_indices']
            owners=saved['slice_series_uids'][indices]
            if not np.all(owners==owners[:,:1]) or not np.all(owners[:,0]==series) or not np.all(saved['same_series']):
                raise ValueError('Cross-series triplet detected')
            if images.dtype!=torch.float16 or images.ndim!=4 or tuple(images.shape[1:])!=(3,384,384) or not len(images):
                raise ValueError('Invalid preprocessed images')
            slots=torch.from_numpy(saved['triplet_slot_ids'])
            present=torch.from_numpy(saved['slot_present'])
        if self.k is not None:
            chosen=torch.randperm(len(images))[:self.k]
            # A short bag stays short; never duplicate triplets to fill k.
            images,slots=images[chosen],slots[chosen]
        item=dict(images=images,triplet_slot_ids=slots,slot_present=present,study_uid=file.stem)
        if self.labels is not None:item['targets']=torch.tensor(self.labels.loc[file.stem].to_numpy(dtype=np.float32))
        return item


In [ ]:
roots=[]
for recipe_file in Path('/kaggle/input').rglob('recipe.json'):
    if json.loads(recipe_file.read_text()).get('format')=='rsna_min_overlap_384_v1':roots.append(recipe_file.parent)
roots=sorted(set(roots));assert len(roots)==32, 'Attach all 32 minimum-overlap dataset V2 shards'
train=MinimumOverlap384Dataset(roots,split='train',k=None)
gold=MinimumOverlap384Dataset(roots,split='held_out_gold',k=None,labels=False)
assert len(train)==4333 and len(gold)==74
assert not {p.stem for p in train.files}&{p.stem for p in gold.files}
sample=train[0]
print(sample['images'].shape,sample['images'].dtype)
rows=[];audits=[]
import hashlib
for root in roots:
    manifest=json.loads((root/'manifest.json').read_text());assert manifest['complete'] and not manifest['failures']
    rows.extend(manifest['completed'])
    for row in manifest['completed']:
        file=root/row['file'];assert file.is_file() and file.stat().st_size==row['bytes']
        with np.load(file,allow_pickle=False) as saved:
            audit=json.loads(str(saved['audit_json']));indices=saved['triplet_indices']
            owners=saved['slice_series_uids'][indices]
            assert np.all(owners==owners[:,:1]) and np.all(saved['same_series'])
            assert len({tuple(t) for t in indices})==len(indices)
            usage=saved['slice_usage_counts'];assert (usage>0).all() and usage.sum()==3*len(indices)
            assert len(usage)==audit['optimal_unique_slices']
            assert 3*len(indices)-len(usage)==audit['minimum_repeated_slice_uses']
            primary=saved['primary_disjoint_subset'];members=indices[primary].ravel()
            assert len(np.unique(members))==len(members) and primary.sum()==audit['optimal_disjoint_subset']
            assert 0<len(indices)<=60 and audit['recipe_sha256']==manifest['recipe_sha256']
            audits.append(audit)
    file=root/manifest['completed'][0]['file']
    with file.open('rb') as stream:assert hashlib.file_digest(stream,'sha256').hexdigest()==manifest['completed'][0]['sha256']
    with np.load(file,allow_pickle=False) as saved:
        assert saved['images'].dtype==np.float16 and saved['images'].shape[1:]==(3,384,384)
        assert np.isfinite(saved['images']).all()
assert len(rows)==4407 and len({r['study_uid'] for r in rows})==4407
summary=dict(attached_shards=32,studies=4407,training_studies=len(train),gold_studies=len(gold),
    triplets=sum(r['windows'] for r in rows),unique_slices=sum(r['source_slices'] for r in rows),
    repeated_slice_uses=sum(a['repeated_slice_uses'] for a in audits),
    zero_overlap_studies=sum(a['repeated_slice_uses']==0 for a in audits),
    short_bags=sum(r['windows']<60 for r in rows),triplets_min=min(r['windows'] for r in rows),
    triplets_max=max(r['windows'] for r in rows),cross_series_triplets=0,
    npz_bytes=sum(r['bytes'] for r in rows),recipe_sha256=audits[0]['recipe_sha256'])
assert summary['triplets']==264323 and summary['unique_slices']==609494
assert summary['zero_overlap_studies']==670 and summary['short_bags']==15
Path('/kaggle/working/dataset_validation.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
